**PP434 portfolio** · Jakob Kilian Bilan · CC7, coordinates map

# Scottish councils: coordinates and income tax

One point per council (longitude, latitude from the course map file), with the income tax due on the council's 2024 annual pay figure under Scotland's 2024-25 bands and under the rest of the UK's. Output: `data/cc7_scotland_councils.csv`.

In [ ]:
import io, json, os
import requests
import pandas as pd

TOPO_URL = "https://raw.githubusercontent.com/jhellingsdata/map-data/refs/heads/main/gbr/LAD_SCT_2026_05.json"
PAY_URL  = "https://raw.githubusercontent.com/jhellingsdata/LSE-PP434/main/weeks/week07/data/wages_LAD_SCT.csv"
YEAR = 2024
OUTPUTS = ["data/cc7_scotland_councils.csv"]

## 1. Tax schedules (2024-25)

In [ ]:
# --- 1. Tax schedules, tax year 2024-25 (non-savings, non-dividend income) -------------------
# Bands are in TAXABLE income (after the personal allowance). Sources: gov.scot factsheet 2024-25;
# gov.uk income tax rates 2024-25.
SCOTLAND = [(2_306, 0.19), (13_991, 0.20), (31_092, 0.21), (62_430, 0.42), (125_140, 0.45), (float("inf"), 0.48)]
REST_OF_UK = [(37_700, 0.20), (125_140, 0.40), (float("inf"), 0.45)]

def personal_allowance(gross):
    pa = 12_570
    if gross > 100_000:                       # tapered: £1 for every £2 over £100,000
        pa = max(0, pa - (gross - 100_000) / 2)
    return pa

def income_tax(gross, bands):
    taxable = max(0, gross - personal_allowance(gross))
    tax, lower = 0.0, 0.0
    for upper, rate in bands:
        if taxable > lower:
            tax += (min(taxable, upper) - lower) * rate
        lower = upper
    return round(tax, 2)

# Hand-checked cases (see README): £35,000 -> Scotland £4,547.33, rUK £4,486.00
assert income_tax(35_000, SCOTLAND) == 4547.33 and income_tax(35_000, REST_OF_UK) == 4486.00

## 2. Coordinates from the map file

In [ ]:
# --- 2. Council points from the course map file (TopoJSON) ------------------------------------
topo = requests.get(TOPO_URL, timeout=30).json()
tf = topo.get("transform")

def decode(arc):
    x = y = 0
    out = []
    for p in arc:
        if tf:                                 # quantised: positions are deltas
            x += p[0]; y += p[1]
            out.append((x * tf["scale"][0] + tf["translate"][0], y * tf["scale"][1] + tf["translate"][1]))
        else:
            out.append((p[0], p[1]))
    return out

arcs = [decode(a) for a in topo["arcs"]]

def ring(indexes):
    pts = []
    for i in indexes:
        seg = arcs[i] if i >= 0 else arcs[~i][::-1]
        pts.extend(seg if not pts else seg[1:])
    return pts

def area_centroid(pts):                         # planar centroid in degrees: fine for a council
    a = cx = cy = 0.0
    for (x0, y0), (x1, y1) in zip(pts, pts[1:] + pts[:1]):
        c = x0 * y1 - x1 * y0
        a += c; cx += (x0 + x1) * c; cy += (y0 + y1) * c
    a /= 2
    return abs(a), cx / (6 * a), cy / (6 * a)

rows = []
for g in topo["objects"]["geog"]["geometries"]:
    polys = [g["arcs"]] if g["type"] == "Polygon" else g["arcs"]
    # use the biggest polygon's outer ring, so island councils get a point on land
    best = max((area_centroid(ring(p[0])) for p in polys), key=lambda t: t[0])
    rows.append({"geo_id": g["properties"]["id"], "name": g["properties"]["name"],
                 "longitude": round(best[1], 4), "latitude": round(best[2], 4)})
points = pd.DataFrame(rows)

## 3. Join pay and compute tax

In [ ]:
# --- 3. Join pay, compute tax under both systems ------------------------------------------------
pay = pd.read_csv(PAY_URL)
pay = pay[pay["date"] == YEAR][["geo_id", "value"]].rename(columns={"value": "pay"})
df = points.merge(pay, on="geo_id", how="left", validate="one_to_one")
df["tax_scotland"] = df["pay"].apply(lambda v: income_tax(v, SCOTLAND))
df["tax_rest_of_uk"] = df["pay"].apply(lambda v: income_tax(v, REST_OF_UK))
df["difference"] = (df["tax_scotland"] - df["tax_rest_of_uk"]).round(2)
df["year"] = YEAR

os.makedirs("data", exist_ok=True)
df.to_csv("data/cc7_scotland_councils.csv", index=False)
print(df.sort_values("difference").to_string(index=False))
print("councils:", len(df), "| missing pay:", df["pay"].isna().sum(),
      "| lon range", df.longitude.min(), df.longitude.max(), "| lat range", df.latitude.min(), df.latitude.max())

In [ ]:
# Colab only: download the output to your computer, then upload it to the data folder of your GitHub repository
try:
    from google.colab import files
    for f in OUTPUTS:
        files.download(f)
except ImportError:
    print("Not running in Colab: the files are in the working folder:", OUTPUTS)